# Ch 16. 위치 인코딩 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 제공한다. 모든 출력은 비워 두었으며 코드 셀은 위에서 아래로 독립적인 CPU 환경에서 실행된다.


## 문제 1

**문제**: Sinusoidal PE의 차원 0과 차원 32의 주파수를 계산하고 비교하라.

### 해설

짝수 차원 $2i$의 각주파수는 $\omega_i=10000^{-2i/d}$이다. $d=64$에서 차원 0은 $i=0$이라 1이고, 차원 32는 $i=16$이라 $10^{-2}=0.01$이다. 따라서 후자는 위치에 따라 100배 느리게 변한다.


In [ ]:
import numpy as np

d_model = 64
dimensions = np.array([0, 32])
frequencies = 10000.0 ** (-dimensions / d_model)
periods = 2 * np.pi / frequencies
assert np.allclose(frequencies, [1.0, 0.01])
print({int(d): {"angular_frequency": float(w), "period": round(float(p), 3)}
       for d, w, p in zip(dimensions, frequencies, periods)})


## 문제 2

**문제**: 학습 가능한 PE로 100 위치를 학습하고, 위치 0~50과 51~99의 차이를 시각화하라.

### 해설

학습 가능한 PE에는 위치 사이의 매끄러움이 자동으로 보장되지 않는다. 재현 가능한 작은 목표 함수를 정해 100개 벡터를 학습한 뒤 두 구간의 평균·분산·인접 차이를 비교해야 하며, 시각적 인상만으로 결론내리지 않는다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Learn 100 position vectors against a fixed smooth target and compare the requested halves.
rng = np.random.default_rng(1602)
position = np.arange(100)[:, None]
frequency = np.arange(1, 9)[None, :] / 20
target = np.sin(position * frequency)
pe = rng.normal(scale=0.5, size=target.shape)
losses = []
for _ in range(120):
    error = pe - target
    losses.append(float(np.mean(error**2)))
    pe -= 0.2 * error
half_gap = float(np.linalg.norm(pe[:50].mean(0) - pe[50:].mean(0)))
adjacent = [float(np.linalg.norm(np.diff(block, axis=0), axis=1).mean()) for block in (pe[:51], pe[50:])]
fig, axis = plt.subplots(figsize=(6, 2.5)); axis.imshow(pe.T, aspect="auto", cmap="coolwarm"); plt.close(fig)
assert losses[-1] < 1e-12 and half_gap > 0
print({"initial_loss": round(losses[0], 5), "final_loss": losses[-1],
       "half_mean_gap": round(half_gap, 4), "adjacent_change": np.round(adjacent, 4).tolist()})


## 문제 3

**문제**: ALiBi의 헤드별 기울기 공식 $m_h = 2^{-8h/H}$를 적용하고, 긴 시퀀스에서의 효과를 시뮬레이션하라.

### 해설

$m_h$는 헤드 번호와 함께 지수적으로 작아진다. 거리 $r$에 대한 편향 $-m_hr$ 때문에 큰 기울기 헤드는 근거리를 강하게 선호하고 작은 기울기 헤드는 장거리 문맥을 보존한다.


In [ ]:
import numpy as np

heads, length = 8, 128
h = np.arange(1, heads + 1)
slopes = 2.0 ** (-8 * h / heads)
distance = np.arange(length)
bias = -slopes[:, None] * distance[None, :]
weights = np.exp(bias - bias.max(axis=1, keepdims=True)); weights /= weights.sum(axis=1, keepdims=True)
expected_distance = weights @ distance
assert np.all(np.diff(slopes) < 0) and np.all(np.diff(expected_distance) > 0)
print({"slopes": slopes.round(6).tolist(), "expected_attended_distance": expected_distance.round(2).tolist()})


## 문제 4

**문제**: RoPE의 "상대적 거리만 의존" 성질을 다양한 위치 쌍에서 검증하라.

### 해설

2차원 회전 블록에서 $(R_mq)^T(R_nk)=q^TR_{n-m}k$이다. 따라서 $(m,n)$을 같은 양만큼 이동해도 내적은 변하지 않는다. 아래 검사는 서로 다른 세 위치 쌍에서 같은 상대 거리의 내적이 일치함을 보인다.


In [ ]:
import numpy as np

rng = np.random.default_rng(1604)
q, k = rng.normal(size=(2, 2))
theta = 0.17
def rotate(vector, position):
    angle = theta * position
    matrix = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    return matrix @ vector
pairs = [(2, 9), (11, 18), (40, 47)]
scores = np.array([rotate(q, m) @ rotate(k, n) for m, n in pairs])
reference = q @ rotate(k, pairs[0][1] - pairs[0][0])
assert np.allclose(scores, reference, atol=1e-12)
print({"relative_distance": 7, "pair_scores": scores.round(10).tolist(), "max_error": float(np.max(np.abs(scores-reference)))})


## 문제 5

**문제**: Position Interpolation의 수식 $m \to m \cdot L_{train}/L_{test}$를 RoPE에 적용하여 외삽 효과를 설명하라.

### 해설

보간 비율 $s=L_{train}/L_{test}<1$을 위치에 곱하면 테스트의 최대 회전각이 학습 범위 안으로 압축된다. 예를 들어 2배 길이에서는 $s=1/2$다. 해상도 손실은 있지만 학습 중 보지 못한 과도한 위상 회전을 피한다.


In [ ]:
import numpy as np

train_length, test_length = 2048, 8192
scale = train_length / test_length
positions = np.array([0, test_length // 2, test_length - 1])
raw_angles = positions.astype(float)
interpolated_angles = positions * scale
assert interpolated_angles.max() < train_length and raw_angles.max() >= train_length
print({"scale": scale, "positions": positions.tolist(), "raw_angles": raw_angles.tolist(),
       "interpolated_angles": interpolated_angles.tolist(), "inside_training_range": True})
